In [6]:
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as md
import re

data_source = {
	# "gemini": "https://www.gemini.com/en-SG/legal/privacy-policy",
	"openai": "https://openai.com/policies/privacy-policy/",
	"anthropic": "https://www.anthropic.com/legal/privacy",
}


def saveHash(key, content):
	sig = hash(content)
	# save to json with key = key, v = hash
	_id = sig
	if _id == sig:
		return True
	# pass
	return False


def splitMarkdown(markdown_text):
	heading_pattern = r"^#{1,6}\s+.*"
	parts = re.split(heading_pattern, markdown_text, flags=re.MULTILINE)
	content_list = [part.strip() for part in parts[1:] if part.strip()]
	return content_list


def removePreamble(markdown_text):
	pattern = r"\A.*?(?=^#\s)"
	cleaned_text = re.sub(pattern, "", markdown_text, flags=re.DOTALL | re.MULTILINE)
	return cleaned_text


def extractContent(url, headers=None):
	if headers is None:
		headers = {
			"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36"
		}

		response = requests.get(url, headers=headers, timeout=10)
		response.raise_for_status()
		html_content = response.text

		soup = BeautifulSoup(html_content, "lxml")

		main_content_element = soup.find("main")

		_markdown_content = md(str(main_content_element), heading_style="ATX")
		markdown_content = removePreamble(_markdown_content)
		return markdown_content


def saveMdFile(content, name):
	if not name.endswith(".md"):
		name = name + ".md"

	with open(name, "w", encoding="utf-8") as file:
		file.write(content)


def collatePolicy(data_source):
	for k, v in data_source.items():
		markdown_content = extractContent(v)
		if saveHash(k, markdown_content):
			# Runs LLM Question analysis of content:
			# adds data and new version questions to question json
			# reanalyses all saved privacy policies against that # (Do at end)
			# updates gui
			pass

		saveMdFile(markdown_content, k)
	return splitMarkdown(markdown_content)


a = collatePolicy(data_source)

FeatureNotFound: Couldn't find a tree builder with the features you requested: lxml. Do you need to install a parser library?